# GRAIL Stage 2b Set-GFlowNet — build-out (Colab Pro+)

Drives the Stage-2b **Set-GFlowNet** milestones end to end: **M0 census gate → M1 small GPU sanity → M2 scale-up (3 seeds)**. Mirrors `reranker_buildout.ipynb`'s Drive-mount / repo / install / data-wiring cells verbatim (proven idioms); only the run cells differ.

**What Set-GFlowNet is for.** Instead of the ensemble's single best-first pass, a trainable policy samples whole *forests* (multi-step reaction trees, depth-capped) and is trained with trajectory-balance (TB) toward a **set-coverage log-reward** over the reranked candidate pool, so it can propose diverse, non-redundant candidate *sets* rather than a single ranked list.

**Honest framing (read before running):**
- **Primary claim (always pursued): diversity/coverage.** The Set-GFlowNet should discover more distinct annotated modes (`modes_discovered`) than a beam/best-first baseline at matched output size `K`, with higher `mean_pairwise_tanimoto` spread / `n_unique_scaffolds` and calibrated `set_size_calibration`. This claim does **not** depend on multi-step chains existing in the data — it holds even if every annotated metabolite is depth-1-reachable.
- **Secondary claim (conditional): multi-step RECALL lift.** Only pursued if the **M0 census** (`scripts/census_multistep.py`, run below) finds a non-trivial fraction of annotated metabolites that are depth-2-reachable but NOT depth-1-reachable (`depth2_only_frac`). If that fraction is ≈ 0 across splits, there is nothing for a depth-2 forest to recall that a depth-1 pass couldn't already reach, so the recall claim is **dropped** and M2's external multi-step recall step is **skipped** — the diversity-only method claim still ships on its own.

**Put these in Google Drive** (dataset + checkpoint are gitignored): the GRAIL repo (push the branch to GitHub or zip to Drive), `train.sdf val.sdf test.sdf` + `*_triples*.txt`, and `artifacts/full5000_priors/checkpoints/generator.pt` (+ `filter.pt`). Edit the **CONFIG** cell, then Run all.

In [ ]:
# 1) Mount Drive + CONFIG (edit these paths)
from google.colab import drive
drive.mount('/content/drive')

GITHUB_URL = ''        # e.g. 'https://github.com/<you>/GRAIL.git'  (leave '' to copy from Drive instead)
BRANCH     = 'metabench-reranker'
DRIVE_REPO = '/content/drive/MyDrive/GRAIL'                 # used only if GITHUB_URL == ''
DRIVE_DATA = '/content/drive/MyDrive/GRAIL_data'            # train.sdf/val.sdf/test.sdf + *_triples*.txt
DRIVE_CKPT = '/content/drive/MyDrive/GRAIL_ckpt'            # generator.pt (+ filter.pt) of full5000_priors
DRIVE_CACHE= '/content/drive/MyDrive/GRAIL_reranker_cache'  # reranker pool cache (survives disconnects; shared with Stage 2a)
TOP_K, MAX_POOL = 200, 150   # matches the reranker's oracle-raising pool size (oracle@15: 100/80 -> 0.372 ; 200/150 -> 0.499)
print('config set; pool', TOP_K, MAX_POOL)

In [ ]:
# 2) Get the repo + install
import subprocess
if GITHUB_URL:
    subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',GITHUB_URL,'/content/GRAIL'], check=True)
else:
    subprocess.run(['cp','-r',DRIVE_REPO,'/content/GRAIL'], check=True)
%cd /content/GRAIL
!pip -q install 'numpy<2' rdkit torch torch_geometric 2>&1 | tail -2
!pip -q install -e . 2>&1 | tail -2
import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 3) Wire Drive data + checkpoint + cache into the repo's expected paths (symlinks)
import os
os.makedirs('grail_metabolism/data', exist_ok=True)
for f in ['train.sdf','val.sdf','test.sdf','train_triples.txt','val_triples.txt','test_triples.txt',
          'train_triples_clean.txt','val_triples_clean.txt','test_triples_clean.txt']:
    src = os.path.join(DRIVE_DATA, f)
    dst = os.path.join('grail_metabolism/data', f)
    if os.path.exists(src) and not os.path.exists(dst): os.symlink(src, dst)
os.makedirs('artifacts/full5000_priors/checkpoints', exist_ok=True)
for f in ['generator.pt','filter.pt']:
    src = os.path.join(DRIVE_CKPT, f)
    dst = os.path.join('artifacts/full5000_priors/checkpoints', f)
    if os.path.exists(src) and not os.path.exists(dst): os.symlink(src, dst)
os.makedirs(DRIVE_CACHE, exist_ok=True)
if not os.path.exists('artifacts/reranker_gate_cache'): os.symlink(DRIVE_CACHE, 'artifacts/reranker_gate_cache')
print('data:', sorted(os.listdir('grail_metabolism/data'))[:6])
print('ckpt:', os.listdir('artifacts/full5000_priors/checkpoints'))
!python -m pytest grail_metabolism/tests/test_set_gflownet.py grail_metabolism/tests/test_gflownet.py grail_metabolism/tests/test_census_multistep.py -q 2>&1 | tail -3

In [ ]:
# 4) M0 census gate: does depth-2 unlock annotated metabolites depth-1 can't reach?
#    Cheap, CPU-only, no training. Run on TEST (the split the final report touches once)
#    and, if a GLORYx substrate set is wired into the repo/data dir, note how to point
#    --split / a custom substrate list at it for a second read of depth2_only_frac.
!python scripts/census_multistep.py --split test --substrates 400 --top-k {TOP_K} --max-pool {MAX_POOL} \
    --out results/census_multistep.json
import json
census = json.load(open('results/census_multistep.json'))
print(json.dumps(census, indent=2))

**M0 gate reading.** `depth2_only_frac` = fraction of annotated metabolites reachable at depth 2 but NOT at depth 1, out of all annotated metabolites seen (`census['depth2_only_frac']`, computed over `census['n_annot']` annotated products across `census['n_subs']` substrates).

- If `depth2_only_frac` ≈ 0 (no depth-2-only recall opportunity) across the substrates censused here → the **multi-step recall claim is dropped**; proceed to M1/M2 for the **diversity-only** method claim, and skip M2's external multi-step recall step.
- If `depth2_only_frac` is non-trivial (there is real depth-2-only recall on the table) → both the diversity claim and the multi-step recall claim are in play for M1/M2, and the external multi-gen recall comparison in M2 is worth running.

(If a GLORYx substrate set is available under `grail_metabolism/data/` or as a separate SMILES list, rerun this cell against it by pointing `--split` at a bundle that resolves to that set, or by adapting `scripts/census_multistep.py --split` plumbing to load it — the script's `--substrates`/`--top-k`/`--max-pool`/`--gen-ckpt`/`--out` flags are otherwise unchanged.)

In [ ]:
# 5) M1 (small GPU sanity): does TB training even work before we scale?
!python -u scripts/run_gflownet.py --train-substrates 300 --max-depth 2 --max-size 10 \
    --epochs 10 --top-k 100 --eval-split val --n-samples 16 --out results/gflownet_m1.json
import json
m1 = json.load(open('results/gflownet_m1.json'))
print(json.dumps(m1['metrics'], indent=2))

**M1 gate (read the live training log above, and `m1['metrics']` printed here):**
- **TB loss decreases and converges.** Scan the `setgfn epoch=N tb_loss=...` lines the cell just printed — loss should trend down and flatten, not diverge or plateau immediately at a high value.
- **`modes_discovered` beats `beam_recall@10`/the beam baseline at matched K=10** (`m1['metrics']['modes_discovered']` vs. the beam recall reported alongside `gflownet_recall@10`).
- **Sampled-set mean log-reward should exceed a random-policy control** — if `mean_pairwise_tanimoto` / `set_size_calibration` look no better than chance (e.g. `modes_discovered` roughly flat vs. beam, or TB loss stuck), that's a fail.

**If the gate fails:** reduce `--max-size`/`--top-k` (shrink the forest/output budget), or lower the learning rate before scaling to M2 (SubTB is noted as a future variance-reduction option, not yet exposed as a config flag). Do not proceed to the 3-seed M2 run on a failing M1.

In [ ]:
# 6) M2 (scale-up, 3 seeds): TEST-split touch-once headline, matched to the reranker's
#    top_k=200/max_pool default pool size. Each seed writes its own results JSON so
#    aggregate_seeds.py can glob them for mean+-std.
import subprocess, json
for seed in [0, 1, 2]:
    out = f'results/gflownet_test_seed{seed}.json' if seed != 0 else 'results/gflownet_test.json'
    subprocess.run(['python','-u','scripts/run_gflownet.py',
                    '--train-substrates','1200','--test-substrates','2000',
                    '--eval-split','test','--max-depth','2','--max-size','15',
                    '--top-k','200','--epochs','20','--n-samples','32',
                    '--seed',str(seed),'--out',out], check=True)
print('--- per-seed TEST headlines ---')
import glob
for f in sorted(glob.glob('results/gflownet_test*.json')):
    m = json.load(open(f))['metrics']
    print(f, '-> gflownet_recall@15', round(m['gflownet_recall@15'], 3),
          '| beam_recall@15', round(m['beam_recall@15'], 3),
          '| modes_discovered', round(m['modes_discovered'], 2))

In [ ]:
# 7) Aggregate the 3 seeds: mean+-std recall + diversity headline (the reportable numbers).
!python scripts/aggregate_seeds.py --glob 'results/gflownet_test*.json'

**M2 notes:**
- This is the TEST split, touched once for the final report (per-seed JSONs are never re-run against TEST after being read).
- The external multi-generation recall comparison (GRAIL Set-GFlowNet vs. other multi-step tools on an external set such as GLORYx) is only worth running if the **M0 gate above found a non-trivial `depth2_only_frac`**; if M0 said diversity-only, skip that external recall step and report `aggregate_seeds.py`'s diversity/coverage table (`modes_discovered`, `mean_pairwise_tanimoto`, `n_unique_scaffolds`, `set_size_calibration`) as the headline, with `gflownet_recall@15` / `beam_recall@15` reported for completeness rather than as the primary claim.

In [ ]:
# 8) Download the results bundle
import shutil
shutil.make_archive('/content/gflownet_results', 'zip', 'results')
from google.colab import files
files.download('/content/gflownet_results.zip')